In [1]:
import gymnasium as gym
import numpy as np
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from rl.agents.dqn import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [2]:
# Wrapper to adjust gymnasium env to keras-rl expected API
class EnvWrapper(gym.Wrapper):
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return obs  # only observation

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return obs, reward, done, info

    def render(self, mode='human', **kwargs):
        return self.env.render()

In [3]:
# Create env
env = EnvWrapper(gym.make('CartPole-v1', render_mode='human'))

states = env.observation_space.shape[0]
actions = env.action_space.n

In [4]:
# Build model
def build_model(states, actions):
    model = Sequential()
    model.add(Flatten(input_shape=(1, states)))  # Window length = 1
    model.add(Dense(24, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(actions, activation='linear'))  # Q-values
    return model

model = build_model(states, actions)


In [5]:
# Build agent
def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(model=model, memory=memory, policy=policy,
                   nb_actions=actions, nb_steps_warmup=10, target_model_update=1e-2)
    return dqn

dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])

In [6]:
# Train agent
dqn.fit(env, nb_steps=50000, visualize=False, verbose=2)

# Test agent
scores = dqn.test(env, nb_episodes=10, visualize=True)
print(f'Average reward: {np.mean(scores.history["episode_reward"])}')

# Save weights
dqn.save_weights('dqn_weights.h5f', overwrite=True)

Training for 50000 steps ...


c:\MLProjects\Reinforcement-Learning-Using-Tensorflow-Keras\.venv38\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,
c:\MLProjects\Reinforcement-Learning-Using-Tensorflow-Keras\.venv38\lib\site-packages\rl\memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


    12/50000: episode: 1, duration: 3.371s, episode steps:  12, steps per second:   4, episode reward: 12.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.667 [0.000, 1.000],  loss: 0.565730, mae: 0.582629, mean_q: -0.035508


c:\MLProjects\Reinforcement-Learning-Using-Tensorflow-Keras\.venv38\lib\site-packages\rl\memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


    33/50000: episode: 2, duration: 0.474s, episode steps:  21, steps per second:  44, episode reward: 21.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.333 [0.000, 1.000],  loss: 0.485802, mae: 0.528410, mean_q: 0.070098
    71/50000: episode: 3, duration: 0.802s, episode steps:  38, steps per second:  47, episode reward: 38.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.658 [0.000, 1.000],  loss: 0.279963, mae: 0.552074, mean_q: 0.425778
    99/50000: episode: 4, duration: 0.595s, episode steps:  28, steps per second:  47, episode reward: 28.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.571 [0.000, 1.000],  loss: 0.119555, mae: 0.605151, mean_q: 0.797707
   110/50000: episode: 5, duration: 0.247s, episode steps:  11, steps per second:  45, episode reward: 11.000, mean reward:  1.000 [ 1.000,  1.000], mean action: 0.364 [0.000, 1.000],  loss: 0.093522, mae: 0.713495, mean_q: 1.090657
   137/50000: episode: 6, duration: 0.576s, episode steps:  27, step

In [7]:
# Reload weights and test again
dqn.load_weights('dqn_weights.h5f')
dqn.test(env, nb_episodes=5, visualize=True)

Testing for 5 episodes ...
Episode 1: reward: 179.000, steps: 179
Episode 2: reward: 185.000, steps: 185
Episode 3: reward: 173.000, steps: 173
Episode 4: reward: 177.000, steps: 177
Episode 5: reward: 183.000, steps: 183
